<a href="https://colab.research.google.com/github/SniAssia/Optimized_data_loaders/blob/main/pipeline_1_ppo_full.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pipeline 1 — Full PPO Training Data Pipeline

Tests the **complete PPO data flow** in order:

| Step | Mode | Produces | Purpose |
|------|------|----------|---------|
| 1 | `sft` | `prompts.bin` + `responses.bin` | Supervised fine-tuning batches |
| 2 | `reward_model` | `prompts.bin` + `chosen.bin` + `rejected.bin` | Train reward model on preferences |
| 3 | `rollout` | `prompts.bin` | Rollout phase — policy generates responses online |

Repo: https://github.com/SniAssia/Optimized_data_loaders/tree/main/reinforcement_learning_data_loader

## 0. Config

In [1]:
import os

REPO_URL = "https://github.com/SniAssia/Optimized_data_loaders.git"
REPO_DIR = "Optimized_data_loaders"
RL_DIR = os.path.join(REPO_DIR, "reinforcement_learning_data_loader")
LIBTORCH_PATH="/content/libtorch"

## 1. Clone / Pull Repo

In [2]:
import subprocess

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

print("Repo contents:")
for f in sorted(os.listdir(RL_DIR)):
    print(" ", f)

Repo contents:
  CMakeLists.txt
  README.md
  collator.h
  dataloader.h
  dataset.h
  distributed.h
  main.cpp
  prefetcher.h
  prepare_dataset.py
  readme2.md
  rollout_buffer.h
  sampler.h
  tokenize_dataset.py


## 2. Download LibTorch (Colab)

In [3]:
import torch
print("PyTorch: ", torch.__version__)
print("CUDA:", torch.version.cuda)

PyTorch: 2.11.0+cu128
CUDA:    12.8


In [4]:
# Only run if libtorch not yet present
if not os.path.isdir(LIBTORCH_PATH):
    !wget -q https://download.pytorch.org/libtorch/nightly/cu128/libtorch-cxx11-abi-shared-with-deps-latest.zip
    !unzip -q libtorch-cxx11-abi-shared-with-deps-latest.zip
    print("libtorch ready at", LIBTORCH_PATH)
else:
    print("libtorch already present at", LIBTORCH_PATH)

libtorch ready at /content/libtorch


## 3. Download Raw Dataset  (Anthropic/hh-rlhf)

In [5]:
jsonl_path = os.path.join(RL_DIR, "hh_rlhf_ppo.jsonl")

result = subprocess.run(
    [
        "python3", "prepare_dataset.py",
        "--dataset", "Anthropic/hh-rlhf",
        "--output",  "hh_rlhf_ppo.jsonl",
    ],
    cwd=RL_DIR, capture_output=True, text=True,
)
print("STDOUT:", result.stdout)
print("STDERR:", result.stderr[-2000:] if result.stderr else "")
result.check_returncode()

STDOUT: wrote 160615 records, skipped 185

STDERR: Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.

Generating train split: 100%|██████████| 160800/160800 [00:02<00:00, 55271.38 examples/s]

Generating test split: 100%|██████████| 8552/8552 [00:00<00:00, 56895.19 examples/s]



## 4. Build C++ Binary (dataloader_demo)

In [7]:
import shutil

def check_tool(name):
    path = shutil.which(name)
    print(f"{name}: {'found → ' + path if path else 'NOT FOUND'}")
    return path is not None

have_cmake = check_tool("cmake")
if not have_cmake:
    subprocess.run(["apt-get", "install", "-y", "-q", "cmake"], check=True)
    have_cmake = check_tool("cmake")

have_gpp   = check_tool("g++") or check_tool("clang++")
libtorch_ok = LIBTORCH_PATH and os.path.isdir(LIBTORCH_PATH)
print(f"libtorch: {'found at ' + LIBTORCH_PATH if libtorch_ok else 'NOT FOUND'}")

can_build = have_cmake and have_gpp and libtorch_ok
print("\ncan_build =", can_build)

cmake: found → /usr/local/bin/cmake
g++: found → /usr/bin/g++
libtorch: found at /content/libtorch

can_build = True


In [8]:
build_dir = os.path.join(RL_DIR, "build")
build_ok  = False

if can_build:
    os.makedirs(build_dir, exist_ok=True)

    cfg = subprocess.run(
        ["cmake", f"-DCMAKE_PREFIX_PATH={LIBTORCH_PATH}", ".."],
        cwd=build_dir, capture_output=True, text=True,
    )
    print("=== cmake configure ===")
    print(cfg.stdout[-2000:])
    if cfg.stderr: print(cfg.stderr[-1000:])

    if cfg.returncode == 0:
        bld = subprocess.run(
            ["cmake", "--build", ".", "-j"],
            cwd=build_dir, capture_output=True, text=True,
        )
        print("=== cmake build ===")
        print(bld.stdout[-2000:])
        if bld.stderr: print(bld.stderr[-1000:])
        build_ok = bld.returncode == 0

print("\nbuild_ok =", build_ok)

=== cmake configure ===
-- The CXX compiler identification is GNU 11.4.0
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- Found CUDA: /usr/local/cuda (found version "12.8") 
-- The CUDA compiler identification is NVIDIA 12.8.93 with host compiler GNU 11.4.0
-- Detecting CUDA compiler ABI info
-- Detecting CUDA compiler ABI info - done
-- Check for working CUDA compiler: /usr/local/cuda/bin/nvcc - skipped
-- Detecting CUDA compile features
-- Detecting CUDA compile features - done
-- Found CUDAToolkit: /usr/local/cuda/include (found version "12.8.93")
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Found Threads: TRUE
-- PyTorch: CUDA detected: 12.8
-- PyTorch: CUDA nvcc is: /usr/local/cuda/bin/nvcc
-- PyTorch: CUDA toolkit directory: /usr/local/cuda
-- PyTorch: Header versio

---
# PPO Phase 1 — SFT Batches

**Purpose:** Supervised Fine-Tuning — teach the base model to follow instructions before RL.

**Batch format:** `(prompt, response)` — model learns to predict response tokens given the prompt.

**Files produced:** `prompts.bin` + `responses.bin`

In [12]:
import json

input_path  = os.path.join(RL_DIR, "hh_rlhf_ppo.jsonl")
output_path = os.path.join(RL_DIR, "hh_rlhf_sft.jsonl")

with open(input_path) as fin, open(output_path, "w") as fout:
    for line in fin:
        rec = json.loads(line)
        rec["response"] = rec["chosen"]   # treat chosen as the supervised response
        fout.write(json.dumps(rec) + "\n")

print("Done — written to hh_rlhf_sft.jsonl")

Done — written to hh_rlhf_sft.jsonl


In [14]:
sft_out = os.path.join(RL_DIR, "out_sft")

result = subprocess.run(
    [
        "python3", "tokenize_dataset.py",
        "--input",      "hh_rlhf_sft.jsonl",
        "--output-dir", "out_sft",
        "--mode",       "sft",
        # sft treats 'chosen' field as the supervised response
    ],
    cwd=RL_DIR, capture_output=True, text=True,
)
print("STDOUT:", result.stdout)
print("STDERR:", result.stderr[-1500:] if result.stderr else "")
result.check_returncode()

STDOUT: [tokenize_dataset] loaded 160615 records from hh_rlhf_sft.jsonl
[tokenize_dataset] loading tokenizer 'inceptionai/jais-family-590m' ...
[tokenize_dataset] tokenizer loaded (eos_token_id=0)
[tokenize_dataset] wrote 160615 sequences to out_sft/prompts.bin
[tokenize_dataset] wrote 160615 sequences to out_sft/responses.bin

STDERR: Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.



In [15]:
# Inspect SFT .bin sizes
for fname in ["prompts.bin", "responses.bin"]:
    fpath = os.path.join(sft_out, fname)
    if os.path.isfile(fpath):
        size_mb = os.path.getsize(fpath) / 1e6
        print(f"{fname}: {size_mb:.2f} MB")
    else:
        print(f"{fname}: NOT FOUND")

prompts.bin: 96.60 MB
responses.bin: 47.44 MB


In [16]:
binary_path = os.path.join(build_dir, "dataloader_demo")

if build_ok and os.path.isfile(binary_path):
    print("========== PPO Phase 1: SFT ==========")
    run = subprocess.run(
        [os.path.abspath(binary_path), "sft"],
        cwd=sft_out, capture_output=True, text=True,
    )
    print(run.stdout)
    if run.returncode != 0:
        print("STDERR:", run.stderr)
else:
    print("Binary not built — see cmake cell above.")

========== PPO Phase 1: SFT ==========
[SFT] 40156 batches
[SFT] batch 0 input_ids=[4, 494] labels=[4, 494]
[SFT] batch 1 input_ids=[4, 63] labels=[4, 63]
[SFT] batch 2 input_ids=[4, 206] labels=[4, 206]



### What SFT Batches Look Like

```
[SFT] batch 0  prompt=[4, 128]  response=[4, 128]  labels=[4, 128]
               ↑ batch_size     ↑ seq_len           ↑ prompt masked to -100
```

- `prompt` — token IDs of the input question
- `response` — token IDs of the full sequence (prompt + answer)
- `labels` — same as response but prompt positions set to `-100` (ignored in cross-entropy loss)

The model is trained to predict only the **answer tokens**, not the prompt.

---
# PPO Phase 2 — Reward Model Training Batches

**Purpose:** Train a reward model to score responses. It learns which response humans prefer.

**Batch format:** `(prompt, chosen, rejected)` — pairs of preferred vs dispreferred responses.

**Files produced:** `prompts.bin` + `chosen.bin` + `rejected.bin`

In [17]:
rm_out = os.path.join(RL_DIR, "out_reward_model")

result = subprocess.run(
    [
        "python3", "tokenize_dataset.py",
        "--input",      "hh_rlhf_ppo.jsonl",
        "--output-dir", "out_reward_model",
        "--mode",       "reward_model",
    ],
    cwd=RL_DIR, capture_output=True, text=True,
)
print("STDOUT:", result.stdout)
print("STDERR:", result.stderr[-1500:] if result.stderr else "")
result.check_returncode()

STDOUT: [tokenize_dataset] loaded 160615 records from hh_rlhf_ppo.jsonl
[tokenize_dataset] loading tokenizer 'inceptionai/jais-family-590m' ...
[tokenize_dataset] tokenizer loaded (eos_token_id=0)
[tokenize_dataset] wrote 160615 sequences to out_reward_model/prompts.bin
[tokenize_dataset] wrote 160615 sequences to out_reward_model/chosen.bin
[tokenize_dataset] wrote 160615 sequences to out_reward_model/rejected.bin
[tokenize_dataset] reward_model mode: prompts.bin / chosen.bin / rejected.bin written to out_reward_model

STDERR: Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.



In [18]:
for fname in ["prompts.bin", "chosen.bin", "rejected.bin"]:
    fpath = os.path.join(rm_out, fname)
    if os.path.isfile(fpath):
        size_mb = os.path.getsize(fpath) / 1e6
        print(f"{fname}: {size_mb:.2f} MB")
    else:
        print(f"{fname}: NOT FOUND")

prompts.bin: 96.60 MB
chosen.bin: 47.44 MB
rejected.bin: 45.22 MB


In [19]:
if build_ok and os.path.isfile(binary_path):
    print("========== PPO Phase 2: Reward Model ==========")
    run = subprocess.run(
        [os.path.abspath(binary_path), "reward_model"],
        cwd=rm_out, capture_output=True, text=True,
    )
    print(run.stdout)
    if run.returncode != 0:
        print("STDERR:", run.stderr)
else:
    print("Binary not built.")

========== PPO Phase 2: Reward Model ==========
[REWARD_MODEL] 40156 batches
[REWARD_MODEL] batch 0 chosen=[4, 358] rejected=[4, 358]
[REWARD_MODEL] batch 1 chosen=[4, 49] rejected=[4, 63]
[REWARD_MODEL] batch 2 chosen=[4, 209] rejected=[4, 182]



### What Reward Model Batches Look Like

```
[REWARD_MODEL] batch 0  chosen=[4, 358]  rejected=[4, 358]
                         ↑ batch_size ↑ seq_len
```

- **No labels** — the reward model reads the full sequence and outputs one scalar per sample
- Chosen and rejected can have **different sequence lengths**
- Loss: `−log(sigmoid(score(chosen) − score(rejected)))`
- After training this model is **frozen** and used as a scorer in Phase 3

---
# PPO Phase 3 — Rollout Batches

**Purpose:** The policy (SFT model) generates responses online. The frozen reward model then scores them.

**Batch format:** `prompts only` — the model generates the responses during training, not from stored data.

**Files produced:** `prompts.bin` only

In [20]:
rollout_out = os.path.join(RL_DIR, "out_rollout")

result = subprocess.run(
    [
        "python3", "tokenize_dataset.py",
        "--input",      "hh_rlhf_ppo.jsonl",
        "--output-dir", "out_rollout",
        "--mode",       "rollout",
    ],
    cwd=RL_DIR, capture_output=True, text=True,
)
print("STDOUT:", result.stdout)
print("STDERR:", result.stderr[-1500:] if result.stderr else "")
result.check_returncode()

STDOUT: [tokenize_dataset] loaded 160615 records from hh_rlhf_ppo.jsonl
[tokenize_dataset] loading tokenizer 'inceptionai/jais-family-590m' ...
[tokenize_dataset] tokenizer loaded (eos_token_id=0)
[tokenize_dataset] wrote 160615 sequences to out_rollout/prompts.bin (5226 truncated)
[tokenize_dataset] rollout mode: prompts.bin written to out_rollout (prompt-only, right-truncated to 512)

STDERR: Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.



In [21]:
for fname in ["prompts.bin"]:
    fpath = os.path.join(rollout_out, fname)
    if os.path.isfile(fpath):
        size_mb = os.path.getsize(fpath) / 1e6
        print(f"{fname}: {size_mb:.2f} MB")
    else:
        print(f"{fname}: NOT FOUND")

prompts.bin: 92.50 MB


In [22]:
if build_ok and os.path.isfile(binary_path):
    print("========== PPO Phase 3: Rollout ==========")
    run = subprocess.run(
        [os.path.abspath(binary_path), "rollout"],
        cwd=rollout_out, capture_output=True, text=True,
    )
    print(run.stdout)
    if run.returncode != 0:
        print("STDERR:", run.stderr)
else:
    print("Binary not built.")

========== PPO Phase 3: Rollout ==========
[ROLLOUT] 40154 prompt batches (group_size=8)
[ROLLOUT] prompt batch 0 prompt_ids=[4, 384]
[ROLLOUT] prompt batch 1 prompt_ids=[4, 384]
[ROLLOUT] buffer sealed: 8 samples, 2 minibatches



### What Rollout Batches Look Like

```
[ROLLOUT] batch 0  prompt=[4, 64]
                   ↑ only prompts — responses are generated online by the policy
```

- Only **prompt token IDs** are provided
- The policy generates a response → frozen reward model scores it → PPO updates policy
- This is the **online** phase — no stored responses needed

---
# Full PPO Pipeline Summary

```
hh_rlhf_ppo.jsonl
       │
       ├─── tokenize (sft)           → out_sft/prompts.bin + responses.bin
       │         ↓
       │    [SFT batch]  prompt=[B,T]  response=[B,T]  labels=[B,T]
       │    → trains base model to follow instructions
       │
       ├─── tokenize (reward_model)  → out_reward_model/prompts.bin + chosen.bin + rejected.bin
       │         ↓
       │    [RM batch]   chosen=[B,T]  rejected=[B,T]
       │    → trains reward model to score responses
       │
       └─── tokenize (rollout)       → out_rollout/prompts.bin
                 ↓
            [Rollout batch]  prompt=[B,T]
            → policy generates responses online, reward model scores, PPO updates
```